# Building a Company Brochure Generator

### BUSINESS CHALLENGE:

Build a tool that creates a short brochure for a company, given its name and website - useful for prospective customers, investors, and recruits.

This extends my Day 1 project (scrape -> translate -> summarize) into a multi-step pipeline:

1. Scrape the company's landing page and its raw links
2. Use an LLM to decide which links are actually relevant (About, Careers, Product pages, etc)
3. Scrape those relevant pages too, and combine everything into one blob of content
4. Use an LLM to write the brochure from that combined content

In this version, I have:
- Reused my own `scrape_article` scraper from Day 1, which has fallbacks (Selenium -> trafilatura -> BeautifulSoup) for pages that are harder to scrape
- Used my Pydantic `Settings` class to load API keys instead of raw `os.getenv`
- Used **Groq's free-tier hosted model** (`llama-3.3-70b-versatile`) for every LLM call, instead of a paid OpenAI model

Target company for this notebook: **NVIDIA** (https://www.nvidia.com/en-us/)

In [1]:
# Imports

from pathlib import Path
import sys

from IPython.display import Markdown, display, update_display

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.services.brochure import create_brochure, stream_brochure

## Confirm settings load correctly

This uses the same `Settings` class from Day 1 - it reads `GROQ_API_KEY` from the `.env` file. If this cell errors, check your `.env` file has `GROQ_API_KEY` set.

In [2]:
from src.config.settings import settings

if settings.groq_api_key and len(settings.groq_api_key) > 10:
    print("Groq API key looks good so far")
else:
    print("There might be a problem with your Groq API key - check your .env file")

Groq API key looks good so far


## Step 1: Fetch all the raw links on NVIDIA's homepage

This is a first look at what I am working with - a real company homepage has a lot of navigation links, most of which aren't useful for a brochure (social media, legal pages, etc).

In [3]:
from src.services.scraper import fetch_links

company_name = "NVIDIA"
company_url = "https://www.nvidia.com/en-us/"

raw_links = fetch_links(company_url)
print(f"Found {len(raw_links)} raw links")
raw_links[:20]

Found 585 raw links


['https://www.nvidia.com',
 '#page-content',
 'https://www.nvidia.com/en-us/',
 'https://www.nvidia.com/en-us/data-center/dgx-cloud/',
 'https://build.nvidia.com/',
 'https://docs.nvidia.com/ngc/latest/ngc-private-registry-user-guide.html',
 'https://www.nvidia.com/en-us/gpu-cloud/',
 'https://www.nvidia.com/en-us/data-center/products/dsx/',
 'https://www.nvidia.com/en-us/data-center/virtual-solutions/',
 'https://www.nvidia.com/en-us/studio/',
 'https://www.nvidia.com/en-us/geforce/broadcasting/',
 'https://www.nvidia.com/en-us/software/nvidia-app/',
 'https://www.nvidia.com/en-us/ai-on-rtx/',
 'https://www.nvidia.com/en-us/geforce/rtx-remix/',
 'https://www.nvidia.com/en-us/software/nvidia-app/g-assist/',
 'https://www.nvidia.com/en-us/data-center/products/',
 'https://www.nvidia.com/en-us/data-center/dgx-platform/',
 'https://www.nvidia.com/en-us/data-center/grace-cpu/',
 'https://www.nvidia.com/en-us/data-center/hgx/',
 'https://www.nvidia.com/en-us/edge-computing/products/igx/']

## Step 2: Use Groq to figure out which links are actually relevant

Rather than hand-writing rules to filter these links, I hand the whole list to `llama-3.3-70b-versatile` and ask it to pick out the About/Careers/Product-type pages and return them as structured JSON. This is a good example of a task that's hard to write by hand but easy for an LLM to reason about.

In [4]:
from src.services.link_selector import select_relevant_links

relevant_links = select_relevant_links(company_url, raw_links)
relevant_links

[{'type': 'about page', 'url': 'https://www.nvidia.com/en-us/about-nvidia/'},
 {'type': 'careers page',
  'url': 'https://www.nvidia.com/en-us/about-nvidia/careers/'},
 {'type': 'news page', 'url': 'https://nvidianews.nvidia.com/'},
 {'type': 'blog page', 'url': 'https://blogs.nvidia.com/'},
 {'type': 'developer page', 'url': 'https://developer.nvidia.com/'},
 {'type': 'research page', 'url': 'https://www.nvidia.com/en-us/research/'},
 {'type': 'sustainability page',
  'url': 'https://www.nvidia.com/en-us/sustainability/'},
 {'type': 'foundation page',
  'url': 'https://www.nvidia.com/en-us/foundation/'},
 {'type': 'investor page',
  'url': 'https://investor.nvidia.com/home/default.aspx'}]

## Step 3: Scrape the landing page + all relevant sub-pages

`gather_company_content` scrapes the homepage, then loops through every relevant link found in Step 2 and scrapes those too, combining everything into one text blob (truncated so it fits comfortably in the model's context).

In [5]:
from src.services.brochure import gather_company_content

combined_content = gather_company_content(company_name, company_url)
print(combined_content[:2000])

## Landing page (https://www.nvidia.com/en-us/)

An NVIDIA Vera Rubin AI factory with over 140 megawatts of compute power based on the NVIDIA DSX platform will be open to every developer.
Japan’s strengths in robotics, manufacturing, automotive, telecommunications, and industrial technology give it a powerful foundation for scaling this next wave of AI.
Discover how leading enterprises and startups are developing AI models and agents designed to serve Japanese users, businesses, and institutions.
The collaboration brings physical AI to next-generation L2++ vehicles, factory robotics, and city-scale urban intelligence.
Get the tools you need to use coding agents to slash development time for building, tuning, and operating vision AI agents for physical spaces.
NVIDIA founder and CEO Jensen Huang takes to Japan, meeting the country’s AI builders, sovereign infrastructure partners and gaming community.
Japan and NVIDIA Launch World's First National AI Infrastructure
Japan’s Leaders Build 

## Step 4: Generate the brochure

One final Groq call takes the combined content and writes it up as a markdown brochure.

In [6]:
brochure = create_brochure(company_name, company_url)
display(Markdown(brochure))

# NVIDIA Company Brochure
## Introduction
NVIDIA is a leader in the field of artificial intelligence, accelerated computing, and graphics processing. Our mission is to drive innovation and deliver cutting-edge technologies that transform industries and improve lives.

## Company Mission
At NVIDIA, we are committed to advancing the field of AI and accelerated computing, and using our technologies to drive positive change in the world. Our goal is to enable developers, researchers, and industries to build innovative solutions that improve performance, efficiency, and sustainability.

## Company Culture
NVIDIA's culture is built on a foundation of innovation, collaboration, and inclusivity. We believe in empowering our employees to think creatively, take risks, and push the boundaries of what is possible. Our diverse and global team is passionate about making a positive impact on the world through technology.

## Products and Solutions
NVIDIA offers a wide range of products and solutions that cater to various industries, including:
* Graphics processing units (GPUs) for gaming, professional visualization, and datacenter applications
* High-performance computing (HPC) solutions for scientific research, AI, and deep learning
* Artificial intelligence (AI) and machine learning (ML) solutions for industries such as healthcare, finance, and transportation
* Autonomous vehicles and robotics solutions for safe and efficient mobility

## Careers
NVIDIA is committed to attracting and retaining top talent from around the world. We offer a range of career opportunities in fields such as engineering, research, sales, and marketing. Our employees enjoy a dynamic and supportive work environment, with opportunities for professional growth and development.

## Research and Development
NVIDIA is dedicated to advancing the state-of-the-art in AI, accelerated computing, and related fields. Our research teams publish papers, participate in conferences, and collaborate with academia and industry partners to drive innovation and breakthroughs.

## Sustainability
At NVIDIA, we recognize the importance of sustainability and environmental responsibility. Our technologies are designed to improve energy efficiency, reduce waste, and promote sustainable practices across industries. We are committed to minimizing our own environmental footprint and promoting a culture of sustainability within our organization.

## Join Us
Whether you are a developer, researcher, or industry leader, we invite you to join us on our mission to shape the future of AI, accelerated computing, and graphics processing. Explore our website, learn about our products and solutions, and discover the many ways you can collaborate with us to drive innovation and positive change.

## Bonus: stream the brochure as it's generated

Same pipeline, but `stream_brochure` yields the response as it's generated so we can update the display live, typewriter-style - matching what my instructor showed in Day 5, but running on Groq's free-tier model instead of a paid OpenAI one.

In [ ]:
display_handle = display(Markdown(""), display_id=True)

for partial_response in stream_brochure(company_name, company_url):
    update_display(Markdown(partial_response), display_id=display_handle.display_id)

## To Sum up

- Reused my Day 1 `scrape_article` scraper and Pydantic `Settings` class - no new scraping logic needed beyond `fetch_links`
- Added a new `link_selector.py` module for LLM-based link filtering (structured JSON output)
- Added a new `brochure.py` module that chains: scrape -> select links -> scrape more -> write brochure
- Used **only Groq's free-tier model** end to end, no paid OpenAI calls
